# exp_016 — best-of-N self-consistency on top of exp_014

Runs N=3 stochastic resamples on candidates (full private set by default), clusters answers by Judger.auto_judge equivalence, votes, and falls back to exp_014's answer if no >=2/3 majority forms.

**Required Kaggle attachments:**
1. Competition dataset (auto-attached)
2. `trevorduong/151b-experiments` (Github branch: `exp/016_best_of_n_rescue`) — provides this folder
3. The exp_014 outputs dataset: slug declared in `experiments/exp_016_best_of_n_rescue/config.json` -> `stage1_dataset_name`
4. The utils dataset that ships `judger.py` + `utils.py`

**Setup:** Save Version → Save & Run All (Commit) with GPU T4 x2. Estimated 3-3.5h for `--private-mode full`.

In [ ]:
# Cell 1: install + libcuda stub (Kaggle-specific)
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0" sympy

import subprocess, os

KNOWN_PATHS = [
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
ldconfig_cands = [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
all_cands = KNOWN_PATHS + ldconfig_cands
libcuda = next((p for p in all_cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in all_cands if os.path.exists(p)), None)

if not libcuda:
    result = subprocess.run(["find", "/usr", "-name", "libcuda.so.1", "-not", "-path", "*/stubs/*"],
                            capture_output=True, text=True)
    found = [p.strip() for p in result.stdout.splitlines() if p.strip()]
    libcuda = found[0] if found else None

assert libcuda, "libcuda.so.1 not found"
print(f"Real libcuda: {libcuda}")

stub_dir = "/kaggle/working/cuda_stubs"
os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
for var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[var] = f"{stub_dir}:{os.environ.get(var, '')}"
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"], check=False)

# Required env vars for vLLM on T4 x2
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print("Install complete. → Run → Restart session, then run the next cell.")

In [ ]:
# Cell 2: locate the exp_016 folder + run the best-of-N driver
import glob, os, sys, json

# Re-expose libcuda stub dir in case of kernel restart
_stub_dir = "/kaggle/working/cuda_stubs"
if os.path.isdir(_stub_dir):
    for _var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        os.environ[_var] = f"{_stub_dir}:{os.environ.get(_var, '')}"
os.environ.setdefault("VLLM_USE_V1", "0")
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

EXPERIMENT = "exp_016_best_of_n_rescue"

# Find the experiment dir in any attached Kaggle dataset
candidates = [
    *glob.glob(f"/kaggle/input/**/experiments/{EXPERIMENT}", recursive=True),
    f"experiments/{EXPERIMENT}",
]
exp_dir = next((p for p in candidates if os.path.isdir(p)), None)
assert exp_dir, f"{EXPERIMENT} folder not found in /kaggle/input/**"
print(f"Found exp dir: {exp_dir}")

# Pass --target private --private-mode full (default per config.json).
# To smoke-test first: append --limit 5 here, swap to --target public.
sys.argv = [
    "run_best_of_n.py",
    "--target", "private",
    "--out-dir", "/kaggle/working",
]

# Make the experiment dir importable
sys.path.insert(0, exp_dir)

# Execute as if invoked from CLI
exec_globals = {"__name__": "__main__", "__file__": os.path.join(exp_dir, "run_best_of_n.py")}
with open(os.path.join(exp_dir, "run_best_of_n.py")) as f:
    exec(compile(f.read(), os.path.join(exp_dir, "run_best_of_n.py"), "exec"), exec_globals)

In [ ]:
# Cell 3: peek at the per-candidate vote stats
import json, os
stats_path = "/kaggle/working/best_of_n_stats.json"
if os.path.exists(stats_path):
    stats = json.load(open(stats_path))
    print(f"Target:                {stats['target']}")
    if stats.get('private_mode'):
        print(f"Private mode:          {stats['private_mode']}")
    print(f"Default candidates:    {stats['n_default_candidates']}")
    print(f"Big-budget candidates: {stats['n_big_candidates']}")
    print(f"Overrides applied:     {stats['flipped']}")
    print(f"Fallbacks to exp_014:  {stats['no_majority_fallback']}")
    flip_pct = stats['flipped'] / max(1, stats['flipped'] + stats['no_majority_fallback'])
    print(f"Override rate:         {flip_pct:.1%}")
else:
    print("No stats file yet — Cell 2 probably hasn't finished.")